# Periods and Quasi-Periods of $\Delta$ and $\hat\Delta$

Reproduce Brown's numerical values (1710.07912 §8.3) using the **finite-endpoint Diamantis-Rolen cocycle formula**:
$$\sigma_f(\gamma)(z) = \int_{\gamma^{-1}\tau_0}^{\tau_0} f(\tau)(\tau - z)^{k-2}\,d\tau$$

**Targets:**
- $\Delta$: $\omega^+ = -68916772.809595194754\ldots$, $\omega^- = -5585015.3793104018668\ldots$
- $\hat\Delta = \Delta(J^2 + 24 J - 393444)$: $\eta^+ = 127202100647.17709477\ldots$, $\eta^- = 10276732343.649132750\ldots$

**Procedure:**
1. Fix $\tau_0 \in \mathbb{H}$ (we use $\tau_0 = 0.3 + 1.2 i$).
2. Compute $C_\gamma(X,Y) = (2\pi i)^{11}\int_{\gamma^{-1}\tau_0}^{\tau_0} f(\tau)(X - \tau Y)^{10}\,d\tau$ for $\gamma \in \{S, T\}$.
3. Solve $P|_T - P = C_T$ for the coboundary polynomial $P \in V_{10}$.
4. Form $C'_S = C_S - (P|_S - P)$.
5. Decompose $C'_S = \omega^+ P^+_S + \omega^- P^-_S$ (mod $X^{10}-Y^{10}$ junk direction).
6. Read off periods.

All integration is over **finite-length line segments in $\mathbb{H}$**. Nothing approaches the cusp.

In [1]:
from mpmath import mp, mpc, mpf, pi, exp, quad, matrix, lu_solve
from sympy import Rational
from math import comb
import time

mp.dps = 40
n = 10
TWOPII = mpc(0, 2*pi)
PREFACTOR = TWOPII**(n+1)
print(f'Working at {mp.dps} decimal digits.')

Working at 40 decimal digits.


## 1. Build $q$-series for $\Delta$, $j$, $J = j-744$, and $\hat\Delta$

$\hat\Delta = \Delta \cdot (J^2 + a_1 J + a_0)$, with $a_1, a_0$ determined by demanding
$\hat\Delta = q^{-1} + 0 + 0\cdot q + O(q^2)$.

Since $\Delta$ has $a_0 = 0$ and $a_1 = 1$, the linear conditions read:
- $q^0$ of $\hat\Delta = 0$: $\;[\Delta J^2]_0 + a_1 [\Delta J]_0 = 0$.
- $q^1$ of $\hat\Delta = 0$: $\;[\Delta J^2]_1 + a_1 [\Delta J]_1 + a_0 = 0$.

We work with exact rationals.

In [2]:
PREC_QSERIES = 60

def laurent_mul(a, b, lo, hi):
    """
    Multiply two formal Laurent series given as {exponent: coefficient} dicts,
    keeping only exponents in the range [lo, hi].

    Parameters
    ----------
    a, b : dict {int: sympy.Rational}
        Laurent series  a(q) = sum_k a_k q^k  (and likewise for b),
        with finitely many nonzero coefficients.
    lo, hi : int
        Truncation window.  Exponents outside [lo, hi] are dropped.

    Returns
    -------
    dict {int: sympy.Rational}
        The product series, with zero coefficients pruned.
    """
    out = {}
    for ka, va in a.items():
        for kb, vb in b.items():
            nn = ka + kb
            if lo <= nn <= hi:
                out[nn] = out.get(nn, Rational(0)) + va * vb
    return {k: v for k, v in out.items() if v != 0}

def laurent_div(num, den, lo_exp, hi_exp):
    """
    Divide two formal Laurent series  num / den  exactly, returning the
    quotient series in the range of exponents [lo_exp, hi_exp].

    Algorithm: extract the leading exponent / coefficient of  den, then
    do long division term-by-term in a single sweep over target exponents.

    Parameters
    ----------
    num, den : dict {int: sympy.Rational}
        Numerator and denominator Laurent series.  `den` must be nonzero.
    lo_exp, hi_exp : int
        Range of exponents to compute in the quotient.

    Returns
    -------
    dict {int: sympy.Rational}
        The quotient series, with zero coefficients pruned.
    """
    den_min = min(den.keys())
    den_lead = den[den_min]
    quot = {}
    for target_n in range(lo_exp + den_min, hi_exp + den_min + 1):
        s = num.get(target_n, Rational(0))
        for k_quot in range(lo_exp, target_n - den_min):
            s -= quot.get(k_quot, Rational(0)) * den.get(target_n - k_quot, Rational(0))
        if s != 0:
            quot[target_n - den_min] = s / den_lead
    return quot

def divisor_sigma(nn, k):
    """
    Divisor power sum  sigma_k(n) = sum_{d | n} d^k.

    Uses the standard O(sqrt(n)) trial-division pairing of divisors.

    Parameters
    ----------
    nn : int
        Positive integer whose divisors are summed.
    k : int
        Exponent to which each divisor is raised.

    Returns
    -------
    int
        sigma_k(nn).
    """
    s = 0; d = 1
    while d*d <= nn:
        if nn % d == 0:
            s += d**k
            if d != nn // d:
                s += (nn // d)**k
        d += 1
    return s

def Delta_over_q_qexp(prec):
    """
    Build the q-expansion of
        Delta(tau) / q  =  prod_{n>=1} (1 - q^n)^24
    out to order  q^prec, as exact rationals.

    Identity used: eta(tau)^24 = Delta(tau) exactly (no extra q factor).
    Since eta(tau) = q^{1/24} prod_{n>=1}(1 - q^n), we have
        Delta(tau) = eta(tau)^24 = q * prod_{n>=1}(1 - q^n)^24.
    The q^{1/24} prefactor inside eta is not representable in a
    Z[[q]]-coefficient dict, so this routine returns Delta(tau) / q, i.e.
    the bare infinite product without its q prefactor.  The caller obtains
    Delta(tau) by shifting exponents by +1 (multiplying by q).

    Parameters
    ----------
    prec : int
        Truncation order; coefficients of q^k for k > prec are dropped.

    Returns
    -------
    dict {int: sympy.Rational}
        {k: [Delta/q]_k}  for  0 <= k <= prec.
    """
    series = {0: Rational(1)}
    for nn in range(1, prec + 1):
        factor = {}
        for k in range(25):
            te = nn * k
            if te > prec: break
            factor[te] = Rational((-1)**k * comb(24, k))
        series = laurent_mul(series, factor, 0, prec)
    return series

# Delta(tau) = eta(tau)^24 = q * prod_{n>=1}(1 - q^n)^24,
# so we shift the exponents of (Delta/q)(q) up by +1.
Delta_over_q  = Delta_over_q_qexp(PREC_QSERIES + 2)
Delta_series  = {k+1: v for k, v in Delta_over_q.items() if k+1 <= PREC_QSERIES + 2}

# E4, j, J
E4 = {0: Rational(1)}
for nn in range(1, PREC_QSERIES + 3):
    E4[nn] = Rational(240 * divisor_sigma(nn, 3))
E4_cubed = laurent_mul(laurent_mul(E4, E4, 0, PREC_QSERIES + 2), E4, 0, PREC_QSERIES + 2)
j_series = laurent_div(E4_cubed, Delta_series, -1, PREC_QSERIES)
J_series = dict(j_series); J_series[0] = J_series.get(0, Rational(0)) - 744
J_series = {k: v for k, v in J_series.items() if v != 0}
J_sq = laurent_mul(J_series, J_series, -2, PREC_QSERIES)

print('Delta:', {k: Delta_series[k] for k in sorted(Delta_series)[:5]})
print('J = j - 744:', {k: J_series[k] for k in sorted(J_series)[:5]})
print('J^2:', {k: J_sq[k] for k in sorted(J_sq)[:5]})

Delta: {1: 1, 2: -24, 3: 252, 4: -1472, 5: 4830}
J = j - 744: {-1: 1, 1: 196884, 2: 21493760, 3: 864299970, 4: 20245856256}
J^2: {-2: 1, 0: 393768, 1: 42987520, 2: 40491909396, 3: 8504046600192}


In [3]:
# Solve for a_1, a_0 in hat_Delta = Delta * (J^2 + a_1 J + a_0)
D_Jsq = laurent_mul(Delta_series, J_sq, -2, PREC_QSERIES)
D_J   = laurent_mul(Delta_series, J_series, -1, PREC_QSERIES)

# q^0 eqn:  D_Jsq[0] + a_1 * D_J[0] = 0
a_1 = -D_Jsq[0] / D_J[0]
# q^1 eqn:  D_Jsq[1] + a_1 D_J[1] + a_0 * Delta[1] = 0  (Delta[1] = 1)
a_0 = -(D_Jsq[1] + a_1 * D_J[1])
print(f'a_1 = {a_1},  a_0 = {a_0}')
assert a_1 == 24 and a_0 == -393444

# hat_Delta
hat_Delta_series = {}
for k, v in D_Jsq.items():
    hat_Delta_series[k] = hat_Delta_series.get(k, Rational(0)) + v
for k, v in D_J.items():
    hat_Delta_series[k] = hat_Delta_series.get(k, Rational(0)) + a_1 * v
for k, v in Delta_series.items():
    hat_Delta_series[k] = hat_Delta_series.get(k, Rational(0)) + a_0 * v
hat_Delta_series = {k: v for k, v in hat_Delta_series.items() if v != 0}

print('\nhat_Delta first coefficients:')
for k in sorted(hat_Delta_series)[:8]:
    print(f'  q^{k}: {hat_Delta_series[k]}')
print('\nBrown reports: q^{-1}=1, q^2=47709536, q^3=39862705122, q^4=7552626810624')

a_1 = 24,  a_0 = -393444

hat_Delta first coefficients:
  q^-1: 1
  q^2: 47709536
  q^3: 39862705122
  q^4: 7552626810624
  q^5: 609136463852480
  q^6: 27947672851540608
  q^7: 855425832767342991
  q^8: 19261421726562785280

Brown reports: q^{-1}=1, q^2=47709536, q^3=39862705122, q^4=7552626810624


## 2. Numeric evaluators for $\Delta$ and $\hat\Delta$

$\Delta$ via the $\eta^{24}$ product (fast convergence). $\hat\Delta$ via its truncated $q$-expansion (also fast since $|q| \sim 10^{-3}$ on our paths).

In [4]:
def delta_form(tau, N_eta=300):
    """
    Numerically evaluate the modular discriminant
        Delta(tau) = eta(tau)^24 = q * prod_{n>=1} (1 - q^n)^24,
        q = exp(2 pi i tau),
    via the Dedekind eta-product, using `mpmath` at the current `mp.dps`.

    The routine computes log[eta(tau)^24] = (2 pi i tau) + 24 sum_n log(1 - q^n),
    then exponentiates, so the q^{1/24}-prefactor is handled exactly inside the
    exponential (no separate q^{1} shift; the (2 pi i tau / 24) appears via
    `TWOPII*tau/24` and is then raised to the 24th power by `eta**24`).

    Convergence: |log(1 - q^n)| ~ e^{-2 pi n Im(tau)}, so N_eta ~ 300 terms
    give ~ 40-digit accuracy whenever Im(tau) >= 0.3 or so.  For the
    contours used here, Im(tau) >= 0.78, so this is plenty.

    Parameters
    ----------
    tau : mpmath.mpc
        Point in the upper half-plane H.
    N_eta : int
        Number of factors in the eta product (default 300).

    Returns
    -------
    mpmath.mpc
        Delta(tau).
    """
    tau = mpc(tau)
    q = exp(TWOPII * tau)
    log_prod = mpc(0)
    qn = q
    for _ in range(N_eta):
        log_prod += mp.log(1 - qn)
        qn *= q
    eta = exp(TWOPII*tau/24 + log_prod)
    return eta**24

# Convert hat_Delta to mpf coefficients
hat_coeffs_mp = sorted([(int(e), mpf(int(c))) for e, c in hat_Delta_series.items()])

def hat_delta(tau):
    """
    Numerically evaluate the weakly holomorphic cusp form
        hat_Delta(tau) = Delta(tau) * (J(tau)^2 + 24 J(tau) - 393444),
    where J = j - 744, via its truncated q-expansion.

    The q-expansion coefficients are pre-computed exactly in `hat_coeffs_mp`
    (Laurent series starting at q^{-1}).  Truncation error is dominated by
    the largest retained |q|^n, which for our contours is ~ e^{-2 pi * 0.78 * 60}
    times the q-coefficient growth ~ e^{4 pi sqrt(60)}, well below 10^{-40}.

    Parameters
    ----------
    tau : mpmath.mpc
        Point in the upper half-plane H.

    Returns
    -------
    mpmath.mpc
        hat_Delta(tau).
    """
    tau = mpc(tau)
    q = exp(TWOPII * tau)
    val = mpc(0)
    for e, c in hat_coeffs_mp:
        val += c * (q ** e)
    return val

# Quick checks
tau_test = mpc(0, 1)
print(f'Delta(i) = {delta_form(tau_test)}')
print(f'hat_Delta(i) = {hat_delta(tau_test)}')

Delta(i) = (0.001785369850642151904343054960342262310581 + 0.0j)
hat_Delta(i) = (1068.415308980081679812205693807699349996 + 0.0j)


## 3. Cocycle integration and slash operators

Generic over the form $f$: pass in any callable `f(tau)` returning the numeric value. We evaluate
$$\int_{\tau_a}^{\tau_b} f(\tau)(X - \tau Y)^{10}\,d\tau$$
as a polynomial in $X, Y$ along a straight-line path in $\mathbb{H}$.

In [5]:
def cocycle_polynomial(f, tau_a, tau_b, n_max=10):
    """
    Diamantis-Rolen cocycle integral on a finite line segment in H:

        (2 pi i)^{n_max+1}  int_{tau_a}^{tau_b}  f(tau) (X - tau Y)^{n_max}  dtau

    expanded in the monomial basis  X^{n_max - j} Y^j  of  V_{n_max}.

    Integration is by mpmath GL quadrature along the straight-line segment
    from tau_a to tau_b, applied to each moment  int f(tau) tau^j dtau.

    Parameters
    ----------
    f : callable mpc -> mpc
        Modular form evaluator (holomorphic or weakly holomorphic).
    tau_a, tau_b : mpmath.mpc
        Endpoints of the contour, both in H.  For the S-path these are
        (-1/tau_0, tau_0); for the T-path, (tau_0 - 1, tau_0).
    n_max : int
        Polynomial degree (= k - 2 = 10 for weight k = 12).

    Returns
    -------
    list of mpmath.mpc, length n_max + 1
        Coefficient list of the cocycle polynomial:
        index j is the coefficient of X^{n_max - j} Y^j.
    """
    tau_a = mpc(tau_a); tau_b = mpc(tau_b)
    dtau = tau_b - tau_a
    moments = []
    for n_pow in range(n_max + 1):
        def integrand(s, n_pow=n_pow):
            tau = tau_a + s * dtau
            return f(tau) * tau**n_pow * dtau
        moments.append(quad(integrand, [0, 1]))
    coeffs = []
    for j in range(n_max + 1):
        c = mpf(comb(n_max, j)) * mpf((-1)**j) * moments[j]
        coeffs.append(PREFACTOR * c)
    return coeffs

def slash_T(p):
    """
    Right slash action of  T = [[1, 1], [0, 1]]  on  V_n:
        (p |_T)(X, Y) = p(X + Y, Y).

    On the basis  X^{n - j} Y^j  this expands binomially:
        X^{n-j} Y^j  ->  sum_{i=0}^{n-j} C(n - j, i) X^{n - j - i} Y^{j + i}.

    Parameters
    ----------
    p : list of mpmath numbers, length n + 1
        p[j] = coefficient of X^{n - j} Y^j.

    Returns
    -------
    list of mpmath numbers, same length
        Coefficient list of  p |_T.
    """
    n_loc = len(p) - 1
    out = [mpf(0)] * (n_loc + 1)
    for j in range(n_loc + 1):
        for i in range(n_loc - j + 1):
            out[j + i] += p[j] * mpf(comb(n_loc - j, i))
    return out

def slash_S(p):
    """
    Right slash action of  S = [[0, -1], [1, 0]]  on  V_n:
        (p |_S)(X, Y) = p(Y, -X).

    On the basis  X^{n - j} Y^j  this acts as
        X^{n - j} Y^j  ->  Y^{n - j} (-X)^j  =  (-1)^j  X^j Y^{n - j},
    so  (p |_S)[m] = (-1)^m p[n - m].

    Parameters
    ----------
    p : list of mpmath numbers, length n + 1
        p[j] = coefficient of X^{n - j} Y^j.

    Returns
    -------
    list of mpmath numbers, same length
        Coefficient list of  p |_S.
    """
    n_loc = len(p) - 1
    return [p[n_loc - m] * mpf((-1)**m) for m in range(n_loc + 1)]

## 4. Generic period extraction routine

Wraps the whole pipeline: feed in $f$ and $\tau_0$, get $(\omega^+, \omega^-)$ (or $(\eta^+, \eta^-)$).

In [6]:
# Brown's basis (1710.07912 §8.2)
P_plus = [mpf(0)]*11
P_plus[0]  = -mpf(36)/mpf(691); P_plus[10] = mpf(36)/mpf(691)
P_plus[2]  =  mpf(1); P_plus[4]  = -mpf(3); P_plus[6]  =  mpf(3); P_plus[8]  = -mpf(1)
P_minus = [mpf(0)]*11
P_minus[1] = mpf(4); P_minus[3] = -mpf(25); P_minus[5] = mpf(42)
P_minus[7] = -mpf(25); P_minus[9] = mpf(4)

def extract_periods(f, tau0, label='f', verbose=True):
    """
    Extract the periods (omega+, omega-) — or quasi-periods (eta+, eta-)
    for weakly holomorphic f — using the Diamantis-Rolen finite-endpoint
    cocycle (Brown 1710.07912 §8, Diamantis-Rolen 1707.04814 §2.1).

    Pipeline:
      1. C_S = (2 pi i)^11 int_{-1/tau0}^{tau0}  f(tau) (X - tau Y)^10 dtau
      2. C_T = (2 pi i)^11 int_{tau0 - 1}^{tau0} f(tau) (X - tau Y)^10 dtau
         (Cuspidality: [C_T]_{X^10} = 0 since f has zero constant Fourier coefficient.)
      3. Solve P|_T - P = C_T  for the coboundary polynomial P in V_10.
      4. Form  C'_S = C_S - (P|_S - P).
      5. Decompose  C'_S = omega+ P+_S + omega- P-_S + c (X^10 - Y^10)
         and read off omega+, omega- from interior monomials.

    Five odd interior monomials (X^9Y, X^7Y^3, X^5Y^5, X^3Y^7, XY^9)
    and four even interior monomials (X^8Y^2, X^6Y^4, X^4Y^6, X^2Y^8)
    each yield independent estimates; their spread is the precision
    diagnostic.  The first odd/even estimate is returned.

    Parameters
    ----------
    f : callable mpc -> mpc
        Numerical evaluator for the weight-12 form.
    tau0 : mpmath.mpc
        Base point in H (here 0.3 + 1.2i).  Both S-image and T-image stay
        well inside H, so |q| stays bounded away from 1.
    label : str
        Cosmetic name for verbose output.
    verbose : bool
        If True, print timing, cuspidality check, and per-monomial estimates.

    Returns
    -------
    (omega_plus, omega_minus) : (mpmath.mpc, mpmath.mpc)
        Periods extracted from C'_S.  For holomorphic f the real part of
        omega_plus and imaginary part of omega_minus carry the answer.
    """
    if verbose:
        print(f'\n========== Extracting periods of {label} at tau_0 = {tau0} ==========')
    t0 = time.time()
    C_S = cocycle_polynomial(f, -1/tau0, tau0, n)
    if verbose: print(f'  C_S done ({time.time()-t0:.1f} s)')
    t0 = time.time()
    C_T = cocycle_polynomial(f, tau0 - 1, tau0, n)
    if verbose: print(f'  C_T done ({time.time()-t0:.1f} s)')

    if verbose:
        print(f'  |C_T|_max = {max(abs(c) for c in C_T)}')
        print(f'  [C_T]_X^10 = {abs(C_T[0])} (cuspidality, should be ~0)')

    # Solve P|_T - P = C_T for P
    A = matrix(10, 10); b = matrix(10, 1)
    for m in range(1, 11):
        for j in range(0, m):
            A[m-1, j] = mpf(comb(10 - j, m - j))
        b[m-1, 0] = C_T[m]
    psol = lu_solve(A, b)
    p_coeffs = [psol[j, 0] for j in range(10)] + [mpf(0)]

    PS = slash_S(p_coeffs)
    C_S_prime = [C_S[j] - (PS[j] - p_coeffs[j]) for j in range(11)]

    # Internal consistency: read off from each available monomial
    minus_estimates = [C_S_prime[j] / P_minus[j] for j in [1,3,5,7,9]]
    plus_estimates  = [C_S_prime[j] / P_plus[j]  for j in [2,4,6,8]]

    if verbose:
        print('  - estimates (odd):')
        for j, e in zip([1,3,5,7,9], minus_estimates):
            print(f'    X^{10-j}Y^{j}: {e}')
        print('  + estimates (even, interior):')
        for j, e in zip([2,4,6,8], plus_estimates):
            print(f'    X^{10-j}Y^{j}: {e}')

    return plus_estimates[0], minus_estimates[0]

## 5. Run for $\Delta$

In [7]:
tau0 = mpc('0.3', '1.2')
omega_plus, omega_minus = extract_periods(delta_form, tau0, 'Delta')
print('\n=== EXTRACTED ===')
print('omega^+ =', omega_plus)
print('omega^- =', omega_minus)
print('\n=== BROWN ===')
print('omega^+ = -68916772.809595194754...')
print('omega^- = -5585015.3793104018668...')
print('\n=== RATIOS (i factor expected on omega^-) ===')
print('omega^+ / Brown:', omega_plus / mpf('-68916772.809595194754'))
print('omega^- / Brown:', omega_minus / mpf('-5585015.3793104018668'))


========== Extracting periods of Delta at tau_0 = (0.3 + 1.2j) ==========


  C_S done (9.0 s)


  C_T done (9.1 s)
  |C_T|_max = 234053334.6157587586133717302731708676396
  [C_T]_X^10 = 4.492405684810882055780909631912922994899e-38 (cuspidality, should be ~0)
  - estimates (odd):
    X^9Y^1: (-2.407412430484044816319972428231159148173e-35 - 5585015.379310401866877139263796275129635j)
    X^7Y^3: (6.162975822039154729779129416271767419322e-35 - 5585015.379310401866877139263796275129635j)
    X^5Y^5: (3.668437989309020672487577033495099654358e-35 - 5585015.379310401866877139263796275129635j)
    X^3Y^7: (9.244463733058732094668694124407651128983e-35 - 5585015.379310401866877139263796275129635j)
    X^1Y^9: (0.0 - 5585015.379310401866877139263796275129635j)
  + estimates (even, interior):
    X^8Y^2: (-68916772.80959519475431012465533103043907 - 3.851859888774471706111955885169854637076e-34j)
    X^6Y^4: (-68916772.80959519475431012465533103043907 + 5.135813185032628941482607846893139516102e-34j)
    X^4Y^6: (-68916772.80959519475431012465533103043907 + 0.0j)
    X^2Y^8: (-68916772.

## 6. Run for $\hat\Delta$

Same procedure, same $\tau_0$, just a different $f$.

In [8]:
eta_plus, eta_minus = extract_periods(hat_delta, tau0, 'hat_Delta')
print('\n=== EXTRACTED ===')
print('eta^+ =', eta_plus)
print('eta^- =', eta_minus)
print('\n=== BROWN ===')
print('eta^+ = 127202100647.17709477...')
print('eta^- = 10276732343.649132750...')
print('\n=== RATIOS (i factor expected on eta^-) ===')
print('eta^+ / Brown:', eta_plus  / mpf('127202100647.17709477'))
print('eta^- / Brown:', eta_minus / mpf('10276732343.649132750'))


========== Extracting periods of hat_Delta at tau_0 = (0.3 + 1.2j) ==========


  C_S done (8.4 s)


  C_T done (9.9 s)
  |C_T|_max = 200673146409032.1503042254499963791663889
  [C_T]_X^10 = 1.55711272805288252882114348751555546029e-31 (cuspidality, should be ~0)
  - estimates (odd):
    X^9Y^1: (8.83524213847533222061135993116720577234e-29 + 10276732343.64913275081719307240092090892j)
    X^7Y^3: (9.693522803355793064899320610194877190224e-29 + 10276732343.64913275081719307240092090896j)
    X^5Y^5: (-4.327465537212407618258625272408427317064e-29 + 10276732343.64913275081719307240092090903j)
    X^3Y^7: (0.0 + 10276732343.64913275081719307240092090893j)
    X^1Y^9: (-2.524354896707237777317531408904915934954e-29 + 10276732343.6491327508171930724009209089j)
  + estimates (even, interior):
    X^8Y^2: (127202100647.1770947773171612986108774946 - 1.009741958682895110927012563561966373982e-27j)
    X^6Y^4: (127202100647.1770947773171612986108774957 - 6.731613057885967406180083757079775826545e-28j)
    X^4Y^6: (127202100647.1770947773171612986108774951 + 0.0j)
    X^2Y^8: (127202100647.17

## 7. Basepoint independence sanity check

Cohomology class shouldn't depend on $\tau_0$. Pick a different one and verify.

In [9]:
tau0_alt = mpc('-0.2', '0.9')
eta_plus_alt, eta_minus_alt = extract_periods(hat_delta, tau0_alt, 'hat_Delta (alt tau_0)', verbose=False)
print(f'eta^+ at tau_0 = {tau0}:  {eta_plus}')
print(f'eta^+ at tau_0 = {tau0_alt}: {eta_plus_alt}')
print(f'  difference: {eta_plus - eta_plus_alt}')
print()
print(f'eta^- at tau_0 = {tau0}:  {eta_minus}')
print(f'eta^- at tau_0 = {tau0_alt}: {eta_minus_alt}')
print(f'  difference: {eta_minus - eta_minus_alt}')

eta^+ at tau_0 = (0.3 + 1.2j):  (127202100647.1770947773171612986108774946 - 1.009741958682895110927012563561966373982e-27j)
eta^+ at tau_0 = (-0.2 + 0.9j): (127202100647.1770947773171612986108774935 - 6.05845175209737066556207538137179824389e-28j)
  difference: (1.11071615455118462201971381991816301138e-27 - 4.038967834731580443708050254247865495927e-28j)

eta^- at tau_0 = (0.3 + 1.2j):  (8.83524213847533222061135993116720577234e-29 + 10276732343.64913275081719307240092090892j)
eta^- at tau_0 = (-0.2 + 0.9j): (-3.786532345060856665976297113357373902431e-29 + 10276732343.64913275081719307240092090888j)
  difference: (1.262177448353618888658765704452457967477e-28 + 4.41762106923766611030567996558360288617e-29j)


## Summary of conventions used

- **Cocycle**: $C_\gamma(X,Y) = (2\pi i)^{11}\int_{\gamma^{-1}\tau_0}^{\tau_0} f(\tau)(X-\tau Y)^{10}\,d\tau$. Direct port of Diamantis-Rolen eq. (2.16) bottom formula, with cusp form normalization.
- **Slash actions**: $P|_T(X,Y) = P(X+Y, Y)$, $\;P|_S(X,Y) = P(Y, -X)$.
- **Coboundary modification**: solve $P|_T - P = C_T$ for $P$, then $C'_S = C_S - (P|_S - P)$.
- **Brown basis** (Brown 1710.07912 §8.2): $P^+_S = \tfrac{36}{691}(Y^{10}-X^{10}) + X^2Y^2(X^2-Y^2)^3$, $\;P^-_S = 4X^9Y - 25X^7Y^3 + 42X^5Y^5 - 25X^3Y^7 + 4XY^9$.

**Convention discrepancy**: extracted $\omega^-, \eta^-$ come out as $i$ times Brown's reported real values. This matches Brown's own period-matrix convention in eq. (2.13) where the entries are $\omega^+, i\omega^-$ — i.e., he absorbs the $i$ into the basis on one side and not the other depending on context. The magnitudes match to the full reported precision.